<a href="https://colab.research.google.com/github/Astronom2617/credit-card-fraud-detection/blob/main/Fraud_02_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fraud Detection Baseline Experiment

## Goal
Build and evaluate the first baselines for credit card fraud detection.
The aim is to establish a clear lower bound and a real ML reference point that any future model must beat.

## Task
Binary classification on a highly imbalanced dataset (fraud ≈ 0.17% of all transactions).

## Target
`Class` — 0 = legitimate transaction, 1 = fraud.

## Input features
- `V1` … `V28` — anonymized PCA features
- `Time` — seconds elapsed since the first transaction in the dataset
- `Amount` — transaction amount
- `Hour` — engineered from `Time` (hour of the day, 0–23)

## Previous step
EDA is done in `Fraud_01_EDA` — class balance, Amount/Time distributions, top discriminative PCA features.

## 1. Load Data

In this section, I load the credit card fraud detection dataset from Kaggle.
The dataset contains 284,807 transactions made by European cardholders, where only 492 are fraudulent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('/content/drive/MyDrive/credit-card-fraud-detection/creditcard.csv')
df.shape

### Dataset Shape

Result: `(284807, 31)`

- 284,807 transactions
- 30 input features + 1 target column (`Class`)

This is a classic strongly imbalanced binary classification problem.

## 2. Train / Test Split

I sort transactions by `Time` and use the first **80%** as the training set and the last **20%** as the test set.

Why time-based split (and not random):
- fraud detection is an online problem — at inference time, the model only sees past transactions
- random split would leak information from the future into the training set
- this setup gives a more honest estimate of real-world performance

In [ ]:
df_sorted = df.sort_values(by='Time')
split_idx = int(len(df) * 0.8)

train = df_sorted.iloc[:split_idx]
test = df_sorted.iloc[split_idx:]

In [ ]:
print(train.shape)
print(test.shape)

print(train['Class'].value_counts())
print(test['Class'].value_counts())

### Split Check

- train size: **227,845** (Class 0: 227,428 · Class 1: 417)
- test size: **56,962**  (Class 0: 56,887  · Class 1: 75)

Both classes are present in both splits.
Fraud share in train ≈ 0.18%, in test ≈ 0.13%. The strong imbalance is preserved on both sides, which makes the comparison between models fair.

## 3. Feature Matrix

I separate the target column (`Class`) from the input features for both train and test sets.

In [ ]:
X_train = train.drop('Class', axis=1)
y_train = train['Class']

X_test = test.drop('Class', axis=1)
y_test = test['Class']

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

## 4. Feature Engineering — Hour

`Time` is given in raw seconds since the first transaction.
Idea: extract the **hour of the day** — fraud activity is non-uniform across the day (confirmed in `Fraud_01_EDA`), so this circular time feature can help linear models.

Formula: `(Time % 86400) // 3600` → integer from 0 to 23.

In [ ]:
X_train['Hour'] = (X_train['Time'] % 86400) // 3600
X_test['Hour']  = (X_test['Time']  % 86400) // 3600

## 5. Scaling

`V1` … `V28` are already PCA components — they are centered and on similar scales, so I leave them as-is.
I only scale the columns that are on a different scale:
- `Amount` — heavy-tailed monetary values
- `Time`   — raw seconds, very large numbers
- `Hour`   — engineered, range 0–23

`StandardScaler` is fitted **only on the train set** to avoid leakage.

In [ ]:
scaler = StandardScaler()

cols_to_scale = ['Amount', 'Time', 'Hour']

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

X_train[cols_to_scale].describe()

### Scaling Check

After `StandardScaler`, the scaled columns have mean ≈ 0 and std ≈ 1 on the train set, as expected.
`Amount` still has a long right tail (max ≈ 78), which is fine — Logistic Regression handles it through the learned weights.

## 6. Naive Baseline — DummyClassifier

The first model is a constant predictor that always returns the majority class (`Class = 0`).
Its only purpose is to give us a **lower bound** for accuracy and to expose the accuracy paradox on imbalanced data.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
print(classification_report(y_test, y_pred_dummy, zero_division=0))

### Naive Baseline — Metrics

```
               precision    recall  f1-score   support
           0       1.00      1.00      1.00     56887
           1       0.00      0.00      0.00        75
    accuracy                           1.00     56962
   macro avg       0.50      0.50      0.50     56962
weighted avg       1.00      1.00      1.00     56962
```

- Accuracy: **99.87%** (looks great, but it's a trap)
- Out of 75 fraud cases in the test set, the model found **zero**
- Precision / Recall / F1 for the fraud class = **0**

This is the accuracy paradox in action: on imbalanced data, accuracy is uninformative.
From now on the real metrics are **Precision, Recall, F1, and PR-AUC for class 1**.
This baseline is the lower bound — any real model must beat it.

## 7. Logistic Regression Baseline

Now I move from a constant predictor to a simple ML model.
Logistic Regression is the classic linear baseline for binary classification — fast to train, easy to interpret, and a fair reference point for more complex models later.

In [ ]:
model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)
print(accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, zero_division=0))

### Logistic Regression — Metrics

```
               precision    recall  f1-score   support
           0       1.00      1.00      1.00     56887
           1       0.86      0.57      0.69        75
    accuracy                           1.00     56962
   macro avg       0.93      0.79      0.84     56962
weighted avg       1.00      1.00      1.00     56962
```

Metrics for the fraud class (`Class = 1`):
- Precision: **0.86**
- Recall: **0.57**
- F1: **0.69**

Out of 75 real fraud cases, the model caught ~43 and missed 32.
Compared to the Naive Baseline this is a huge step forward — the model actually detects fraud instead of labeling everything as legitimate.
The main weakness is **low recall**: 43% of fraudulent transactions still slip through, which is not acceptable for a production fraud detection system.

## 8. Final Comparison

All baselines are evaluated on the **same time-based test split** (last 20% of transactions, 56,962 rows, 75 fraud cases).

### Summary Table

| Model | Accuracy | Precision (fraud) | Recall (fraud) | F1 (fraud) | Macro F1 |
|---|---|---|---|---|---|
| DummyClassifier (most_frequent) | 0.9987 | 0.00 | 0.00 | 0.00 | 0.50 |
| Logistic Regression             | 0.9993 | 0.86 | 0.57 | 0.69 | 0.84 |

### Observations

- **Accuracy is misleading.** Both models look ~99.9% accurate, but only one actually detects fraud.
- **Logistic Regression already wins decisively** on every fraud-class metric (Precision, Recall, F1).
- The main bottleneck is **Recall** — too many fraudsters still pass through undetected.

## 9. Next Step → `Fraud_03_Models`

The next notebook in this series will be **`Fraud_03_Models`**, where I push Recall up without destroying Precision:

- **`class_weight='balanced'`** in Logistic Regression — penalize the minority class harder so the model stops ignoring fraud
- **Random Forest** — a stronger non-linear baseline that handles imbalance better out of the box
- **SMOTE** — synthetic oversampling of the minority class (applied to the train set only, never to test)
- evaluation on the **same time-based split** as this notebook, so all numbers stay directly comparable
- **PR-AUC** as the primary metric instead of ROC-AUC, since PR-AUC is more informative under heavy class imbalance